# FHIR Zerobus Endpoint Testing with Service Principal

This notebook tests the FastAPI FHIR ingestion endpoint using **service principal authentication**.

## What This Notebook Does

1. **Authenticates** using service principal credentials from `fhir_zerobus_credentials` scope
2. **Obtains OAuth token** from Databricks workspace
3. **Tests the FastAPI endpoint** with Bearer token authentication
4. **Posts FHIR bundles** to Unity Catalog via Zerobus streaming
5. **Verifies data** was written to Unity Catalog

## Authentication Method

**Service Principal (OAuth M2M)**
* Client ID & Secret stored in Databricks secrets
* Token obtained from workspace OAuth endpoint
* Suitable for production, automation, and API integrations
* More secure than personal access tokens

## App.py Fixes Applied

### Fix 1: Databricks Apps Authentication
* Changed from `Authorization` header to `x-forwarded-*` headers
* Databricks Apps Gateway validates tokens and forwards user identity

### Fix 2: JSON Serialization
* Convert FHIR payload dict to JSON string for Zerobus
* Unity Catalog VARIANT column requires string format

## How to Use This Notebook

1. **First run:** Run all cells to test current state
2. **After app.py changes:** Redeploy the Databricks App
3. **Verify fixes:** Run [Quick Test](#cell-6b1a1fa7-2f8f-4298-8088-874944779ce0) cell
4. **Check data:** Run [Verify Data](#cell-ccd97aa1-f3dd-40c9-afa6-6200bc059f86) cell

## Endpoint Details

* **URL:** `https://fhir-zerobus-ingest-himss2026-7474657999482942.aws.databricksapps.com`
* **Path:** `/api/v1/ingest/fhir-bundle`
* **Method:** POST
* **Auth:** Bearer token (service principal)
* **Content-Type:** application/json

In [0]:
import requests
import json

# Get service principal credentials from secrets
client_id = dbutils.secrets.get(scope="fhir_zerobus_credentials", key="client_id")
client_secret = dbutils.secrets.get(scope="fhir_zerobus_credentials", key="client_secret")

# Get workspace URL for OAuth token endpoint
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
workspace_url = f"https://{workspace_url}"

print(f"Workspace URL: {workspace_url}")
print(f"Client ID: {client_id[:10]}...\n")

# Step 1: Get OAuth token using service principal credentials
print("Step 1: Obtaining OAuth token for service principal...")
token_endpoint = f"{workspace_url}/oidc/v1/token"

try:
    token_response = requests.post(
        token_endpoint,
        data={
            "grant_type": "client_credentials",
            "scope": "all-apis"
        },
        auth=(client_id, client_secret)
    )
    
    if token_response.status_code == 200:
        token_data = token_response.json()
        access_token = token_data.get("access_token")
        print(f"✓ Token obtained successfully")
        print(f"   Token: {access_token[:20]}...\n")
    else:
        print(f"✗ Failed to get token: {token_response.status_code}")
        print(f"   Response: {token_response.text}\n")
        access_token = None
except Exception as e:
    print(f"✗ Error getting token: {e}\n")
    access_token = None

# Step 2: Test FHIR endpoint with service principal token
if access_token:
    # Define the endpoint
    base_url = "https://fhir-zerobus-ingest-himss2026-7474657999482942.aws.databricksapps.com"
    endpoint = f"{base_url}/api/v1/ingest/fhir-bundle"
    
    # Create a basic FHIR Bundle JSON
    fhir_bundle = {
        "resourceType": "Bundle",
        "type": "transaction",
        "entry": [
            {
                "resource": {
                    "resourceType": "Patient",
                    "id": "example-patient-sp-001",
                    "name": [
                        {
                            "use": "official",
                            "family": "ServicePrincipal",
                            "given": ["Test"]
                        }
                    ],
                    "gender": "other",
                    "birthDate": "1990-01-01"
                },
                "request": {
                    "method": "POST",
                    "url": "Patient"
                }
            }
        ]
    }
    
    print("Step 2: Testing FastAPI endpoints with service principal token\n")
    print("="*60)
    
    # Test 1: Root endpoint
    print("\n1. Testing root endpoint (/)")
    try:
        response = requests.get(base_url, headers={"Authorization": f"Bearer {access_token}"})
        print(f"   Status: {response.status_code}")
        print(f"   Response: {response.text[:200]}")
    except Exception as e:
        print(f"   Error: {e}")
    
    # Test 2: Docs endpoint
    print("\n2. Testing /docs endpoint")
    try:
        response = requests.get(f"{base_url}/docs", headers={"Authorization": f"Bearer {access_token}"})
        print(f"   Status: {response.status_code}")
        print(f"   Accessible: {response.status_code == 200}")
    except Exception as e:
        print(f"   Error: {e}")
    
    # Test 3: Health check endpoint
    print("\n3. Testing /health endpoint")
    try:
        response = requests.get(f"{base_url}/health", headers={"Authorization": f"Bearer {access_token}"})
        print(f"   Status: {response.status_code}")
        if response.status_code == 200:
            health_data = response.json()
            print(f"   Health: {health_data.get('status')}")
            print(f"   Zerobus Stream: {health_data.get('zerobus_stream')}")
    except Exception as e:
        print(f"   Error: {e}")
    
    # Test 4: POST FHIR Bundle with service principal token
    print("\n4. POST FHIR Bundle with service principal authentication")
    response = requests.post(
        endpoint,
        json=fhir_bundle,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {access_token}"
        }
    )
    print(f"   Status: {response.status_code}")
    print(f"   Response: {response.text}")
    
    # Test 5: POST without authentication (for comparison)
    print("\n5. POST without authentication (expected to fail)")
    response_no_auth = requests.post(
        endpoint,
        json=fhir_bundle,
        headers={"Content-Type": "application/json"}
    )
    print(f"   Status: {response_no_auth.status_code}")
    print(f"   Response: {response_no_auth.text[:100]}...")
    
    print("\n" + "="*60)
    if response.status_code == 200:
        print("✓ SUCCESS! FHIR Bundle posted successfully with service principal")
        result = response.json()
        print(f"\nBundle UUID: {result.get('bundle_uuid')}")
        print(f"User: {result.get('user')}")
        print(f"Timestamp: {result.get('timestamp')}")
    else:
        print("⚠ REQUEST FAILED")
        print(f"\nStatus: {response.status_code}")
        print(f"Details: {response.text}")
        print("\nNote: After updating app.py, you need to redeploy the Databricks App.")
else:
    print("Cannot test endpoint - failed to obtain service principal token")

## Fixes Applied to app.py

### 1. Authentication Fix ✓
**Problem:** Databricks Apps Gateway strips the `Authorization` header after validation

**Solution:** Updated `verify_databricks_auth()` to read from:
* `x-forwarded-user` - authenticated user identity  
* `x-forwarded-access-token` - user's access token

**Result:** Service principal authentication now works correctly

### 2. VARIANT Column Format Fix ✓
**Problem:** Zerobus treats VARIANT as "JSON stored as string"

**Solution:** Use `json.dumps()` to convert payload to JSON string:
```python
fhir_json_string = json.dumps(payload, ensure_ascii=True, separators=(',', ':'))

record = {
    "bundle_uuid": bundle_uuid,
    "fhir": fhir_json_string,  # JSON string for VARIANT
    "source_system": app.title,
    "event_timestamp": timestamp,
    "user_email": user_email,
}
```

### 3. Timestamp Format Fix ✓
**Problem:** ISO format with `+00:00` can cause parsing issues

**Solution:** Use RFC3339 with `Z` suffix:
```python
timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
```

### 4. Unicode Character Fix ✓
**Problem:** Unicode arrow `→` in app title caused "invalid digit found in string" error

**Solution:** 
* Changed app title from `"FHIR → Zerobus Ingest App"` to `"FHIR to Zerobus Ingest App"`
* Use `ensure_ascii=True` in json.dumps to escape any unicode in FHIR data

**Why:** Zerobus/Delta requires clean ASCII JSON - unicode must be escaped as `\uXXXX`

### Next Steps

1. **Redeploy the Databricks App** - this restarts the Zerobus stream
2. Run [Quick Test](#cell-6b1a1fa7-2f8f-4298-8088-874944779ce0) to verify
3. Expected: **200 OK** with bundle UUID ✓

### Table Schema
* `bundle_uuid`: STRING  
* `fhir`: **VARIANT** (JSON as string)
* `source_system`: STRING
* `event_timestamp`: TIMESTAMP
* `user_email`: STRING

In [0]:
import requests
import json
from datetime import datetime

# Get service principal token
client_id = dbutils.secrets.get(scope="fhir_zerobus_credentials", key="client_id")
client_secret = dbutils.secrets.get(scope="fhir_zerobus_credentials", key="client_secret")
workspace_url = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"

print("Getting service principal token...")
token_response = requests.post(
    f"{workspace_url}/oidc/v1/token",
    data={"grant_type": "client_credentials", "scope": "all-apis"},
    auth=(client_id, client_secret)
)
access_token = token_response.json().get("access_token")
print(f"✓ Token obtained\n")

# Test endpoint
endpoint = "https://fhir-zerobus-ingest-himss2026-7474657999482942.aws.databricksapps.com/api/v1/ingest/fhir-bundle"

# Create test FHIR bundle
test_bundle = {
    "resourceType": "Bundle",
    "type": "transaction",
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "entry": [
        {
            "resource": {
                "resourceType": "Patient",
                "id": f"test-patient-{datetime.utcnow().strftime('%Y%m%d%H%M%S')}",
                "name": [{"family": "TestPatient", "given": ["API"]}],
                "gender": "unknown",
                "birthDate": "2000-01-01"
            },
            "request": {"method": "POST", "url": "Patient"}
        }
    ]
}

print("Posting FHIR Bundle...")
response = requests.post(
    endpoint,
    json=test_bundle,
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}"
    }
)

print(f"Status: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print("✓✓✓ SUCCESS! Both fixes working! ✓✓✓")
    print(f"\n  Bundle UUID: {result['bundle_uuid']}")
    print(f"  User: {result['user']}")
    print(f"  Timestamp: {result['timestamp']}")
    print(f"  Status: {result['status']}")
    print("\nFHIR bundle successfully written to Unity Catalog!")
else:
    print(f"✗ FAILED: {response.status_code}")
    print(f"\nResponse: {response.text}")
    print("\n⚠ Make sure you've redeployed the Databricks App with the updated app.py")

In [0]:
import json
from datetime import datetime, timezone
import uuid

# Simulate exactly what app.py does
test_bundle = {
    "resourceType": "Bundle",
    "type": "transaction",
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "entry": [
        {
            "resource": {
                "resourceType": "Patient",
                "id": f"test-patient-{datetime.utcnow().strftime('%Y%m%d%H%M%S')}",
                "name": [{"family": "TestPatient", "given": ["API"]}],
                "gender": "unknown",
                "birthDate": "2000-01-01"
            },
            "request": {"method": "POST", "url": "Patient"}
        }
    ]
}

# Simulate app.py record creation
bundle_uuid = str(uuid.uuid4())
timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

print("Testing different serialization methods:\n")
print("="*60)

# Method 1: ensure_ascii=False (current app.py)
print("\n1. ensure_ascii=False, separators=(',', ':')")
fhir_json_1 = json.dumps(test_bundle, ensure_ascii=False, separators=(',', ':'))
record_1 = {
    "bundle_uuid": bundle_uuid,
    "fhir": fhir_json_1,
    "source_system": "FHIR → Zerobus Ingest App",
    "event_timestamp": timestamp,
    "user_email": "test@example.com",
}
serialized_1 = json.dumps(record_1, ensure_ascii=False, separators=(',', ':'))
print(f"Length: {len(serialized_1)}")
print(f"Char at 497: '{serialized_1[497] if len(serialized_1) > 497 else 'N/A'}'")
if len(serialized_1) > 497:
    print(f"Context [487:507]: {serialized_1[487:507]}")
print(f"\nFirst 500 chars:\n{serialized_1[:500]}")

# Method 2: Default json.dumps
print("\n" + "="*60)
print("\n2. Default json.dumps (with spaces)")
fhir_json_2 = json.dumps(test_bundle)
record_2 = {
    "bundle_uuid": bundle_uuid,
    "fhir": fhir_json_2,
    "source_system": "FHIR → Zerobus Ingest App",
    "event_timestamp": timestamp,
    "user_email": "test@example.com",
}
serialized_2 = json.dumps(record_2)
print(f"Length: {len(serialized_2)}")
print(f"Char at 497: '{serialized_2[497] if len(serialized_2) > 497 else 'N/A'}'")
if len(serialized_2) > 497:
    print(f"Context [487:507]: {serialized_2[487:507]}")

# Method 3: ensure_ascii=True (safe mode)
print("\n" + "="*60)
print("\n3. ensure_ascii=True (safe mode)")
fhir_json_3 = json.dumps(test_bundle, ensure_ascii=True, separators=(',', ':'))
record_3 = {
    "bundle_uuid": bundle_uuid,
    "fhir": fhir_json_3,
    "source_system": "FHIR → Zerobus Ingest App",
    "event_timestamp": timestamp,
    "user_email": "test@example.com",
}
serialized_3 = json.dumps(record_3, ensure_ascii=True, separators=(',', ':'))
print(f"Length: {len(serialized_3)}")
print(f"Char at 497: '{serialized_3[497] if len(serialized_3) > 497 else 'N/A'}'")
if len(serialized_3) > 497:
    print(f"Context [487:507]: {serialized_3[487:507]}")

print("\n" + "="*60)
print("\nLooking for special characters around position 497...")
for method, serialized in [("Method 1", serialized_1), ("Method 2", serialized_2), ("Method 3", serialized_3)]:
    if len(serialized) > 497:
        print(f"\n{method}:")
        # Check for unicode/special chars
        for i in range(max(0, 490), min(len(serialized), 505)):
            c = serialized[i]
            if ord(c) > 127:
                print(f"  Position {i}: '{c}' (ord: {ord(c)}) - NON-ASCII")
            elif c in ['→', '←', '↔']:
                print(f"  Position {i}: '{c}' (ord: {ord(c)}) - ARROW CHARACTER")

In [0]:
import requests
import json
from datetime import datetime

# Get token
client_id = dbutils.secrets.get(scope="fhir_zerobus_credentials", key="client_id")
client_secret = dbutils.secrets.get(scope="fhir_zerobus_credentials", key="client_secret")
workspace_url = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"

token_response = requests.post(
    f"{workspace_url}/oidc/v1/token",
    data={"grant_type": "client_credentials", "scope": "all-apis"},
    auth=(client_id, client_secret)
)
access_token = token_response.json().get("access_token")

endpoint = "https://fhir-zerobus-ingest-himss2026-7474657999482942.aws.databricksapps.com/api/v1/ingest/fhir-bundle"

# Test with MINIMAL bundle - no timestamp, no special chars
minimal_bundle = {
    "resourceType": "Bundle",
    "type": "transaction",
    "entry": []
}

print("Test 1: Minimal bundle (empty)")
print(f"Payload: {json.dumps(minimal_bundle)}\n")

response = requests.post(
    endpoint,
    json=minimal_bundle,
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {access_token}"}
)

print(f"Status: {response.status_code}")
print(f"Response: {response.text[:300]}\n")

if response.status_code == 200:
    print("✓ Minimal bundle works!")
else:
    print("✗ Minimal bundle fails")
    
# Test 2: Bundle with simple entry (no timestamp in bundle)
print("\n" + "="*60)
print("\nTest 2: Simple patient (no bundle timestamp)")

simple_bundle = {
    "resourceType": "Bundle",
    "type": "transaction",
    "entry": [{
        "resource": {
            "resourceType": "Patient",
            "id": "simple-test-001",
            "name": [{"family": "Test", "given": ["Simple"]}]
        },
        "request": {"method": "POST", "url": "Patient"}
    }]
}

print(f"Payload length: {len(json.dumps(simple_bundle))} chars\n")

response2 = requests.post(
    endpoint,
    json=simple_bundle,
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {access_token}"}
)

print(f"Status: {response2.status_code}")
print(f"Response: {response2.text[:300]}\n")

if response2.status_code == 200:
    print("✓ Simple bundle works!")
    result = response2.json()
    print(f"Bundle UUID: {result['bundle_uuid']}")
else:
    print("✗ Simple bundle fails")

## ✅ Root Cause Found: Unicode Character Issue

### The Problem

**Error:** "invalid digit found in string at line 1 column 497"

**Root Cause:** Unicode arrow character `→` (U+2192) in app title
```python
app = FastAPI(title="FHIR → Zerobus Ingest App")  # ✗ Contains unicode
```

This character was in the `source_system` field:
```json
{"source_system": "FHIR → Zerobus Ingest App"}
```

When serialized with `ensure_ascii=False`, the unicode arrow remained in the JSON, causing Zerobus/Delta parser to fail with "invalid digit" error.

### The Fix

**Two changes in app.py:**

1. **Remove unicode from app title:**
```python
app = FastAPI(title="FHIR to Zerobus Ingest App")  # ✓ ASCII only
```

2. **Use ASCII-safe JSON serialization:**
```python
fhir_json_string = json.dumps(payload, ensure_ascii=True, separators=(',', ':'))
```

With `ensure_ascii=True`, any unicode characters in FHIR data are automatically escaped (e.g., `é` becomes `\u00e9`).

### Why This Matters

* **Zerobus/Delta expects clean ASCII JSON** for VARIANT columns
* Unicode characters must be escaped using `\uXXXX` notation
* The error "invalid digit found in string" was misleading - it was actually a unicode parsing issue

### Next Steps

1. **Redeploy the Databricks App** (this restarts the Zerobus stream)
2. **Run the Quick Test** to verify the fix
3. **Success expected:** 200 OK with bundle UUID

### Summary of All Fixes

| Issue | Fix |
|-------|-----|
| 401 Unauthorized | Use `x-forwarded-*` headers ✓ |
| VARIANT expects string | Use `json.dumps(payload)` ✓ |
| Timestamp format | Use RFC3339 with `Z` suffix ✓ |
| **Unicode parsing error** | **Remove unicode, use `ensure_ascii=True`** ✓ |

## 🚀 All Fixes Applied - Ready to Deploy!

### What Was Fixed

✅ **Authentication:** Using `x-forwarded-*` headers  
✅ **VARIANT Format:** JSON string with `json.dumps()`  
✅ **Timestamp:** RFC3339 format (`2026-03-04T19:30:16Z`)  
✅ **Unicode Issue:** Removed `→` character, using `ensure_ascii=True`

### Deploy Now

**Option 1: Using Databricks CLI**
```bash
databricks bundle deploy -t himss2026
```

**Option 2: Using Databricks Asset Bundles UI**
1. Navigate to your workspace files
2. Find the app bundle configuration
3. Click "Deploy"

### After Deployment

1. **Wait 30-60 seconds** for app to restart and Zerobus stream to initialize
2. **Run [Quick Test](#cell-6b1a1fa7-2f8f-4298-8088-874944779ce0)** cell below
3. **Expected Result:**
   ```
   ✓✓✓ SUCCESS! Both fixes working! ✓✓✓
   
   Bundle UUID: <uuid>
   User: <service-principal-app-id>
   Timestamp: 2026-03-04T19:30:16Z
   Status: ok
   ```

4. **Verify data:** Run [Verify Data in Unity Catalog](#cell-ccd97aa1-f3dd-40c9-afa6-6200bc059f86)

### Why It Will Work Now

The "invalid digit found in string" error was caused by the unicode arrow character `→` in the app title. When this was serialized into JSON with `ensure_ascii=False`, it created invalid JSON for Zerobus/Delta parser.

**Before:**
```json
{"source_system":"FHIR → Zerobus Ingest App"}  // ✗ Unicode breaks parser
```

**After:**
```json
{"source_system":"FHIR to Zerobus Ingest App"}  // ✓ Clean ASCII
```

Plus using `ensure_ascii=True` ensures any unicode in FHIR data is properly escaped as `\uXXXX`.

In [0]:
# Query the most recent FHIR bundles from Unity Catalog
# Note: Update the table name to match your configuration

# Try to find the correct table
try:
    # Check if we can find the table from common naming patterns
    tables_to_try = [
        "main.fhir.fhir_bundles",
        "main.default.fhir_bundles",
        "himss.fhir.fhir_bundles",
        "main.fhir.fhir_bundle_zerobus"
    ]
    
    table_found = None
    for table_name in tables_to_try:
        try:
            df = spark.sql(f"SELECT COUNT(*) as count FROM {table_name}")
            count = df.collect()[0]['count']
            print(f"✓ Found table: {table_name} ({count} records)")
            table_found = table_name
            break
        except:
            continue
    
    if table_found:
        print(f"\nQuerying most recent records from {table_found}...\n")
        
        # Get the latest 5 records
        recent_df = spark.sql(f"""
            SELECT 
                bundle_uuid,
                source_system,
                event_timestamp,
                user_email,
                fhir:resourceType as resource_type,
                fhir:type as bundle_type
            FROM {table_found}
            ORDER BY event_timestamp DESC
            LIMIT 5
        """)
        
        display(recent_df)
        
        print("\n" + "="*60)
        print("✓ Data verification complete!")
        print("\nTo view full FHIR bundle JSON, run:")
        print(f"  spark.sql('SELECT * FROM {table_found} WHERE bundle_uuid = <uuid>').display()")
    else:
        print("⚠ Could not find the FHIR bundles table.")
        print("\nPlease check the table name in your config.py file.")
        print("Common locations:")
        for table in tables_to_try:
            print(f"  - {table}")
        
except Exception as e:
    print(f"✗ Error querying table: {e}")
    print("\nMake sure:")
    print("  1. The table has been created")
    print("  2. You have SELECT permissions on the table")
    print("  3. The table name in config.py matches your Unity Catalog structure")